# 1. Merge & Pre-processing
The cleaned transactions (`train`) and the cleaned customer table (`clientes`) are merged into one dataset, split into training and validation folds, and pre-processed in order: outliers, missing values, and encoding and scaling.

In [13]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.compose import ColumnTransformer
from sklearn.experimental import enable_iterative_imputer  # noqa: F401 (needed for IterativeImputer)
from sklearn.impute import IterativeImputer, KNNImputer, SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score
from sklearn.model_selection import StratifiedShuffleSplit, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

# Same seed everywhere, so every run gives the same splits and draws
RANDOM_STATE = 42

____________________________________________________________________________________________________

## 1.1. Data Loading

In [14]:
trainFile = "../../data/trainDataset/train_2_4_1.csv"
clientFile = "../../data/clientesDataset/clientes_clean.csv"
testFile = "../../data/test.csv"

train = pd.read_csv(trainFile)
clientes = pd.read_csv(clientFile)
test = pd.read_csv(testFile)

RANDOM_STATE= 42

____________________________________________________________________________________________________

## 1.2. Merging the Datasets

In [15]:
train_clientes = train.merge(clientes, on="customer_id", how="left", validate="many_to_one")
assert len(train_clientes) == len(train)

train_clientes.head()

,Total a Pagar,Total Bruto,Total Liquido,Total Imp_,Total Portes,Total Desc_ Global,Total Desc_ Linha,Custo,N_ Detalhes,invoice_count_current_purchase,...,island,municipality,city,postal_area,postal_code,street,address,company_name,customer_since_year,since_2015_or_earlier
0,558.11,510.90,453.75,104.36,0.0,0.0,57.15,153.099911,4.0,1.0,...,(mainland),Municipality MU89: Council of Last Minute Edits,City CY286: Cached Result Falls,Postal area PA228: Hidden Sheet Sector,Postal code PC444: #REF! Delivery Route,Street ST303: Absolute Reference Terrace,Address AD761: Cell Without a Home,Excel Is Not A Database Ltd 1013,2024,0
1,978.80,830.02,795.77,183.03,0.0,0.0,34.25,472.943134,32.0,1.0,...,Island IS8: The Pivot Archipelago,Municipality MU11: Council of Last Minute Edits,City CY147: VLOOKUP Bay,Postal area PA260: Hidden Sheet Sector,Postal code PC636: #REF! Delivery Route,NaN,Address AD382: Cell Without a Home,Excel Is Not A Database Ltd 35,2015,1
2,10.33,12.00,8.40,1.93,0.0,0.0,3.60,6.097485,1.0,1.0,...,(mainland),Municipality MU70: Council of Last Minute Edits,City CY265: Where-less Query Falls,Postal area PA84: Hidden Sheet Sector,Postal code PC200: #REF! Delivery Route,NaN,Address AD723: Cell Without a Home,Excel Is Not A Database Ltd 826,2017,0
3,856.83,696.61,696.61,160.22,0.0,0.0,0.00,408.167275,9.0,1.0,...,Island IS7: The Pivot Archipelago,Municipality MU12: Council of Last Minute Edits,NaN,Postal area PA253: Hidden Sheet Sector,Postal code PC558: #REF! Delivery Route,Street ST144: Last Save Court,Address AD399: Cell Without a Home,Excel Is Not A Database Ltd 530,2018,0
4,1616.10,1488.85,1313.90,302.20,0.0,0.0,174.95,485.401864,33.0,1.0,...,(mainland),Municipality MU142: Council of Last Minute Edits,City CY359: Spill Error Ridge,Postal area PA136: Hidden Sheet Sector,Postal code PC300: #REF! Delivery Route,Street ST395: AutoFill Path,Address AD928: Cell Without a Home,Excel Is Not A Database Ltd 220,2017,0


**Conclusions:**
* **Lossless merge:** every one of the **6,469** transactions found its customer (`validate="many_to_one"` confirms one row per `customer_id` in `clientes`), so no rows were lost or duplicated. The merged table has **57 columns**: the 45 of `train` plus 12 customer attributes.
* **Customer coverage:** the transactions belong to **607** customers, so 536 of the 1,143 customers in `clientes` have no purchase in `train` and are simply not used.
* **New missing values:** the customer attributes bring their own gaps into the merged table, mostly in identifier-like columns: `street` (58.9%), `city` (32.9%) and `postal_code` (3.4%). The descriptive ones are almost complete: `postal_area` (1.0%), `commercial_region` (0.5%), `island` and `municipality` (0.1%). These are handled together with the `train` gaps in the next sections.

____________________________________________________________________________________________________

## 1.3. Feature Type Separation
Imputation and encoding work differently on numbers and on labels (a median or a KNN fill needs numbers, a mode or an `"Unknown"` level works on labels), so the merged columns are split into groups first:

1. **Target:** `repeat_purchase_90d`, the variable to predict.
2. **Identifiers:** `ID` (one per row) and `customer_id` (the join key, kept to group the cross-validation folds by customer).
3. **Excluded:** `company_name`, `address`, `street` and `postal_code` are distinct for almost every customer, so they name a customer instead of describing one; `random_noise` carries no information by construction.
4. **Numeric** and **categorical** features: everything else, separated by data type.

In [16]:
target = "repeat_purchase_90d"
identifiers = ["ID", "customer_id"]

# Unique per customer (they name a customer instead of describing one) or pure noise
excluded = ["company_name", "address", "street", "postal_code", "random_noise"]

features = [c for c in train_clientes.columns if c not in [target] + identifiers + excluded]

# Automatic separation between numerical and categorical variables
numeric = [c for c in features if pd.api.types.is_numeric_dtype(train_clientes[c])]
categorical = [c for c in features if c not in numeric]

# Every column belongs to exactly one group
assert set([target] + identifiers + excluded + numeric + categorical) == set(train_clientes.columns)

X = train_clientes[numeric + categorical]
y = train_clientes[target]

print(f"{len(numeric)} numeric | {len(categorical)} categorical | {len(identifiers)} identifiers | {len(excluded)} excluded")
print("categorical:", categorical)

37 numeric | 12 categorical | 2 identifiers | 5 excluded
categorical: ['payment_type', 'salesperson', 'commercial_zone', 'transport_method', 'warehouse', 'document_series', 'country_or_region', 'commercial_region', 'island', 'municipality', 'city', 'postal_area']


**Conclusions:**
* **49 features** are left for the model: **37 numeric** and **12 categorical**. The target has no missing values and is fairly balanced (56% of purchases are repeated within 90 days).
* **Numbers that are labels:** some numeric columns are categories written as numbers: `purchase_month`, `purchase_dayofweek`, `purchase_quarter` and the 0/1 flags `is_first_purchase`, `is_weekend` and `since_2015_or_earlier`. None of them has missing values, so they do not affect the imputation; whether they are encoded as categories is decided in the encoding step.

____________________________________________________________________________________________________

## 1.4. Missing Values Overview

In this step, we measure the missing values, which gave us some information about which features still have gaps after the merge, how many, and whether if a value is missing says something about the target.

**Conclusions:**

____________________________________________________________________________________________________

## 1.5. Train / Validation Split

In [17]:
splits = StratifiedShuffleSplit(n_splits=20, test_size=0.2,
                                random_state=RANDOM_STATE)

# The 20 (training rows, validation rows) pairs, reused by every comparison below
X_splits = list(splits.split(X, y))

**Conclusions:**

____________________________________________________________________________________________________

## 1.6. Outliers

### 1.6.1. Outlier Detection

### 1.6.2. Outlier Treatment

**Conclusions:**

____________________________________________________________________________________________________

## 1.7. Missing Values Comparison and Selection

### 1.7.1. Numerical Features

### 1.7.2. Categorical Features

### 1.7.3. Final Imputation

**Conclusions:**

____________________________________________________________________________________________________

## 1.8. Encoding and Scaling

### 1.8.1. Encoding

### 1.8.2. Scaling

**Conclusions:**